题目地址：https://tianchi.aliyun.com/competition/entrance/231784/information

软件环境：Windows11、Python 3.14.4、numpy 2.4.4、scikit-learn 1.8.0、pandas 3.0.2、SciPy 1.17.1

硬件环境：13th Gen Core(TM) i5-13500H（12核、16线程）、16GB物理内存、60GB虚拟内存

每个模型训练限时：15min

In [134]:
from pathlib import Path
import time
import winsound
import pickle

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import TargetEncoder, OneHotEncoder


In [135]:
train = pd.read_csv(
    'used_car_train_20200313.csv',
    sep=' ',
    parse_dates=['creatDate'],
    date_format={"creatDate": "%Y%m%d"}
)

train

,SaleID,name,regDate,model,brand,bodyType,fuelType,gearbox,power,kilometer,...,v_5,v_6,v_7,v_8,v_9,v_10,v_11,v_12,v_13,v_14
0,0,736,20040402,30.0,6,1.0,0.0,0.0,60,12.5,...,0.235676,0.101988,0.129549,0.022816,0.097462,-2.881803,2.804097,-2.420821,0.795292,0.914762
1,1,2262,20030301,40.0,1,2.0,0.0,0.0,0,15.0,...,0.264777,0.121004,0.135731,0.026597,0.020582,-4.900482,2.096338,-1.030483,-1.722674,0.245522
2,2,14874,20040403,115.0,15,1.0,0.0,0.0,163,12.5,...,0.251410,0.114912,0.165147,0.062173,0.027075,-4.846749,1.803559,1.565330,-0.832687,-0.229963
3,3,71865,19960908,109.0,10,0.0,0.0,1.0,193,15.0,...,0.274293,0.110300,0.121964,0.033395,0.000000,-4.509599,1.285940,-0.501868,-2.438353,-0.478699
4,4,111080,20120103,110.0,5,1.0,0.0,0.0,68,5.0,...,0.228036,0.073205,0.091880,0.078819,0.121534,-1.896240,0.910783,0.931110,2.834518,1.923482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,149995,163978,20000607,121.0,10,4.0,0.0,1.0,163,15.0,...,0.280264,0.000310,0.048441,0.071158,0.019174,1.988114,-2.983973,0.589167,-1.304370,-0.302592
149996,149996,184535,20091102,116.0,11,0.0,0.0,0.0,125,10.0,...,0.253217,0.000777,0.084079,0.099681,0.079371,1.839166,-2.774615,2.553994,0.924196,-0.272160
149997,149997,147587,20101003,60.0,11,1.0,1.0,0.0,90,6.0,...,0.233353,0.000705,0.118872,0.100118,0.097914,2.439812,-1.630677,2.290197,1.891922,0.414931
149998,149998,45907,20060312,34.0,10,3.0,1.0,0.0,156,15.0,...,0.256369,0.000252,0.081479,0.083558,0.081498,2.075380,-2.633719,1.414937,0.431981,-1.659014


# 数据清洗

In [136]:
def clean_df(df):
    df['regDate'] = pd.to_datetime(df['regDate'], format='%Y%m%d', errors='coerce')
    # 注意：Excel的基准通常被视为 1899-12-30
    base_date = pd.Timestamp('1899-12-30')
    # 计算天数差并转换为整数
    df['regDate'] = (df['regDate'] - base_date).dt.days

    df['creatDate'] = (df['creatDate'] - base_date).dt.days

    df['notRepairedDamage'] = df['notRepairedDamage'].replace('-', np.nan).astype(float)

    df['age'] = (df['creatDate'] - df['regDate']) / 365.25

    df.loc[df['power'] > 600, 'power'] = np.nan

    df.drop(columns=['SaleID', 'name', 'offerType', 'seller'], inplace=True)  # offerType全为0、seller几乎全为0）

    return df


train = clean_df(train)

X_train = train.drop(columns=['price'])
y_train = train['price']

In [137]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 28 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   regDate            138653 non-null  float64
 1   model              149999 non-null  float64
 2   brand              150000 non-null  int64  
 3   bodyType           145494 non-null  float64
 4   fuelType           141320 non-null  float64
 5   gearbox            144019 non-null  float64
 6   power              149857 non-null  float64
 7   kilometer          150000 non-null  float64
 8   notRepairedDamage  125676 non-null  float64
 9   regionCode         150000 non-null  int64  
 10  creatDate          150000 non-null  int64  
 11  price              150000 non-null  int64  
 12  v_0                150000 non-null  float64
 13  v_1                150000 non-null  float64
 14  v_2                150000 non-null  float64
 15  v_3                150000 non-null  float64
 16  v_4          

In [138]:
train.describe()

,regDate,model,brand,bodyType,fuelType,gearbox,power,kilometer,notRepairedDamage,regionCode,...,v_6,v_7,v_8,v_9,v_10,v_11,v_12,v_13,v_14,age
count,138653.000000,149999.000000,150000.000000,145494.000000,141320.000000,144019.000000,149857.000000,150000.000000,125676.000000,150000.000000,...,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,138653.000000
mean,38017.717453,47.129021,8.052733,1.792369,0.375842,0.224943,116.399941,12.597160,0.113904,2583.077267,...,0.044923,0.124692,0.058144,0.061996,-0.001000,0.009035,0.004813,0.000313,-0.000688,12.134380
std,1953.203543,49.536040,7.864956,1.760640,0.548677,0.417546,68.500152,3.919576,0.317696,1885.363218,...,0.051743,0.201410,0.029186,0.035692,3.772386,3.286071,2.517478,1.288988,1.038685,5.347576
min,33239.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-9.168192,-5.558207,-9.639552,-4.153899,-6.546556,0.240931
25%,36529.000000,10.000000,1.000000,0.000000,0.000000,0.000000,75.000000,12.500000,0.000000,1018.000000,...,0.000038,0.062474,0.035334,0.033930,-3.722303,-1.951543,-1.871846,-1.057789,-0.437034,8.052019
50%,38027.000000,30.000000,6.000000,1.000000,0.000000,0.000000,110.000000,15.000000,0.000000,2196.000000,...,0.000812,0.095866,0.057014,0.058484,1.624076,-0.358053,-0.130753,-0.036245,0.141246,12.095825
75%,39511.000000,66.000000,13.000000,3.000000,1.000000,0.000000,150.000000,15.000000,0.000000,3843.000000,...,0.102009,0.125243,0.079382,0.087491,2.844357,1.255022,1.776933,0.942813,0.680378,16.227242
max,42350.000000,247.000000,39.000000,7.000000,6.000000,1.000000,600.000000,15.000000,1.000000,8120.000000,...,0.151420,1.404936,0.160791,0.222787,12.357011,18.819042,13.847792,11.147669,8.658418,25.248460


In [139]:
train

,regDate,model,brand,bodyType,fuelType,gearbox,power,kilometer,notRepairedDamage,regionCode,...,v_6,v_7,v_8,v_9,v_10,v_11,v_12,v_13,v_14,age
0,38079.0,30.0,6,1.0,0.0,0.0,60.0,12.5,0.0,1046,...,0.101988,0.129549,0.022816,0.097462,-2.881803,2.804097,-2.420821,0.795292,0.914762,12.005476
1,37681.0,40.0,1,2.0,0.0,0.0,0.0,15.0,NaN,4366,...,0.121004,0.135731,0.026597,0.020582,-4.900482,2.096338,-1.030483,-1.722674,0.245522,13.023956
2,38080.0,115.0,15,1.0,0.0,0.0,163.0,12.5,0.0,2806,...,0.114912,0.165147,0.062173,0.027075,-4.846749,1.803559,1.565330,-0.832687,-0.229963,11.997262
3,35316.0,109.0,10,0.0,0.0,1.0,193.0,15.0,0.0,434,...,0.110300,0.121964,0.033395,0.000000,-4.509599,1.285940,-0.501868,-2.438353,-0.478699,19.507187
4,40911.0,110.0,5,1.0,0.0,0.0,68.0,5.0,0.0,6977,...,0.073205,0.091880,0.078819,0.121534,-1.896240,0.910783,0.931110,2.834518,1.923482,4.191650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,36684.0,121.0,10,4.0,0.0,1.0,163.0,15.0,0.0,4576,...,0.000310,0.048441,0.071158,0.019174,1.988114,-2.983973,0.589167,-1.304370,-0.302592,15.802875
149996,40119.0,116.0,11,0.0,0.0,0.0,125.0,10.0,0.0,2826,...,0.000777,0.084079,0.099681,0.079371,1.839166,-2.774615,2.553994,0.924196,-0.272160,6.357290
149997,40454.0,60.0,11,1.0,1.0,0.0,90.0,6.0,0.0,3302,...,0.000705,0.118872,0.100118,0.097914,2.439812,-1.630677,2.290197,1.891922,0.414931,5.483915
149998,38788.0,34.0,10,3.0,1.0,0.0,156.0,15.0,0.0,1877,...,0.000252,0.081479,0.083558,0.081498,2.075380,-2.633719,1.414937,0.431981,-1.659014,10.056126


In [140]:
train.nunique()

regDate                3597
model                   248
brand                    40
bodyType                  8
fuelType                  7
gearbox                   2
power                   445
kilometer                13
notRepairedDamage         2
regionCode             7905
creatDate                96
price                  3763
v_0                  143997
v_1                  143998
v_2                  143997
v_3                  143998
v_4                  143998
v_5                  139624
v_6                  109766
v_7                  138709
v_8                  142451
v_9                  140617
v_10                 143997
v_11                 143997
v_12                 143997
v_13                 143998
v_14                 143998
age                    9042
dtype: int64

# 结果记录函数

In [141]:

def record(pkl_file, random_search, start_time, end_time):
    print(f'{pkl_file}估计器的平均交叉验证分数:{random_search.best_score_:.4f}')
    print(f'训练耗时:{(end_time - start_time) / 60:.2f}min')

    with open(pkl_file, 'wb') as f:
        pickle.dump(random_search.best_estimator_, f)
        print(f"{pkl_file}已保存")

    # 完成声音提醒
    winsound.PlaySound("SystemAsterisk", winsound.SND_ALIAS)



# 模型训练

## HistGradientBoostingRegressor

### 数据预处理

In [142]:
cat_features = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage', 'model', 'brand']
TE_features = ['regionCode']

cat_target_transformer = Pipeline(steps=[
    ("target", TargetEncoder(target_type='continuous', random_state=0)),
])

processor1 = ColumnTransformer(
    transformers=[
        ('TE_preprocess', cat_target_transformer, TE_features),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')



### 默认参数

In [143]:
pkl_file = 'HistGradientBoostingRegressor-basic.pkl'

if not Path(pkl_file).exists():
    start_time = time.time()

    HGBR = make_pipeline(
        processor1,
        HistGradientBoostingRegressor(
            categorical_features=[name for name in cat_features],
            random_state=0
        )
    )

    param_dist = {}

    random_search = RandomizedSearchCV(
        HGBR,
        param_dist,
        n_iter=1,
        scoring='neg_mean_absolute_error',
        random_state=0,
        n_jobs=-1,
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record(pkl_file, random_search, start_time, end_time)

### 最佳参数

In [144]:
pkl_file = 'HistGradientBoostingRegressor-tuned.pkl'

if not Path(pkl_file).exists():
    start_time = time.time()

    HGBR = make_pipeline(
        processor1,
        HistGradientBoostingRegressor(
            categorical_features=[name for name in cat_features],
            random_state=0
        )
    )

    param_dist = {
        'histgradientboostingregressor__learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'histgradientboostingregressor__max_iter': [100, 200, 300, 500],
        'histgradientboostingregressor__max_leaf_nodes': [31, 63, 127, 255],
        'histgradientboostingregressor__min_samples_leaf': [5, 10, 20, 50],
        'histgradientboostingregressor__l2_regularization': [0.0, 0.1, 1.0, 5.0, 10.0],
    }

    random_search = RandomizedSearchCV(
        HGBR,
        param_dist,
        n_iter=30,
        scoring='neg_mean_absolute_error',
        random_state=0,
        n_jobs=-1,
        error_score='raise'
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record(pkl_file, random_search, start_time, end_time)

## RandomForestRegressor

### 数据预处理

In [145]:
OH_features = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage']
TE_features = ['model', 'brand', 'regionCode']

cat_OH_transformer = Pipeline(steps=[
    ("OH", OneHotEncoder(handle_unknown='infrequent_if_exist', sparse_output=False)),
])
cat_TE_transformer = Pipeline(steps=[
    ("target", TargetEncoder(target_type='continuous', random_state=0)),
])

processor2 = ColumnTransformer(
    transformers=[
        ('OH_preprocess', cat_OH_transformer, OH_features),
        ('TE_preprocess', cat_TE_transformer, TE_features),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')



### 默认参数

In [146]:
pkl_file = 'RandomForestRegressor-basic.pkl'

if not Path(pkl_file).exists():
    start_time = time.time()

    RF = make_pipeline(
        processor2,
        RandomForestRegressor(
            random_state=0,
            n_jobs=-1
        )
    )

    param_dist = {}

    random_search = RandomizedSearchCV(
        RF,
        param_dist,
        n_iter=1,
        scoring='neg_mean_absolute_error',
        random_state=0,
        n_jobs=-1,
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record(pkl_file, random_search, start_time, end_time)

### 最佳参数

In [147]:
pkl_file = 'RandomForestRegressor-tuned.pkl'

if not Path(pkl_file).exists():
    start_time = time.time()

    RF = make_pipeline(
        processor2,
        RandomForestRegressor(
            random_state=0,
            n_jobs=-1
        )
    )

    param_dist = {
        'randomforestregressor__n_estimators': [50, 100, 150, 200],
        'randomforestregressor__max_depth': [10, 20, 30, None],
        'randomforestregressor__min_samples_split': [2, 5, 10],
        'randomforestregressor__min_samples_leaf': [1, 2, 4],
        'randomforestregressor__max_features': ['sqrt', 'log2', 0.5, 0.7],
    }

    random_search = RandomizedSearchCV(
        RF,
        param_dist,
        n_iter=20,
        scoring='neg_mean_absolute_error',
        cv=3,
        random_state=0,
        n_jobs=-1,
        error_score='raise',
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record(pkl_file, random_search, start_time, end_time)

## ExtraTreesRegressor

### 数据预处理


In [148]:
OH_features = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage']
TE_features = ['model', 'brand', 'regionCode']

cat_OH_transformer = Pipeline(steps=[
    ("OH", OneHotEncoder(handle_unknown='infrequent_if_exist', sparse_output=False)),
])
cat_TE_transformer = Pipeline(steps=[
    ("target", TargetEncoder(target_type='continuous', random_state=0)),
])

processor3 = ColumnTransformer(
    transformers=[
        ('OH_preprocess', cat_OH_transformer, OH_features),
        ('TE_preprocess', cat_TE_transformer, TE_features),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

### 默认参数

In [149]:
pkl_file = 'ExtraTreesRegressor-basic.pkl'

if not Path(pkl_file).exists():
    start_time = time.time()

    ETR = make_pipeline(
        processor3,
        ExtraTreesRegressor(
            random_state=0,
            n_jobs=-1
        )
    )

    param_dist = {}

    random_search = RandomizedSearchCV(
        ETR,
        param_dist,
        n_iter=1,
        scoring='neg_mean_absolute_error',
        random_state=0,
        n_jobs=-1,
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record(pkl_file, random_search, start_time, end_time)

### 最佳参数

In [150]:
pkl_file = 'ExtraTreesRegressor-tuned.pkl'

if not Path(pkl_file).exists():
    start_time = time.time()

    ETR = make_pipeline(
        processor3,
        ExtraTreesRegressor(
            random_state=0,
            n_jobs=-1
        )
    )

    param_dist = {
        'extratreesregressor__n_estimators': [100, 200, 300],
        'extratreesregressor__max_depth': [15, 20, 25, 30, None],
        'extratreesregressor__min_samples_split': [2, 5, 10],
        'extratreesregressor__min_samples_leaf': [1, 2, 4],
        'extratreesregressor__max_features': ['sqrt', 'log2', 0.5, 0.7],
    }

    random_search = RandomizedSearchCV(
        ETR,
        param_dist,
        n_iter=25,
        cv=3,
        scoring='neg_mean_absolute_error',
        random_state=0,
        n_jobs=-1,
        error_score='raise',
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record(pkl_file, random_search, start_time, end_time)

# 生成答案

HistGradientBoostingRegressor-basic.pkl估计器的平均交叉验证分数:-683.7885

训练耗时:0.15min

HistGradientBoostingRegressor-tuned.pkl估计器的平均交叉验证分数:-563.9358

训练耗时:3.20min

RandomForestRegressor-basic.pkl估计器的平均交叉验证分数:-604.7148

训练耗时:3.72min

RandomForestRegressor-tuned.pkl估计器的平均交叉验证分数:-597.7662

训练耗时:16.88min

ExtraTreesRegressor-basic.pkl估计器的平均交叉验证分数:-562.6159

训练耗时:1.60min

ExtraTreesRegressor-tuned.pkl估计器的平均交叉验证分数:-574.3355

训练耗时:14.04min

In [151]:
# X_test = pd.read_csv(
#     'used_car_testA_20200313.csv',
#     sep=' ',
#     parse_dates=['creatDate'],
#     date_format={"creatDate": "%Y%m%d"}
# )
# X_test = clean_df(X_test)
#
# with open('ExtraTreesRegressor-basic.pkl', 'rb') as f:
#     HGBR_model = pickle.load(f)
#
#     scores = HGBR_model.predict(X_test)
#
#     answers = pd.DataFrame({
#         'SaleID': range(150000, 150000 + len(X_test)),
#         'price': scores,
#     })
#     answers.to_csv('answer.csv', index=False)
